<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/WithoutKD_corn_B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip Corn.zip

Streaming output truncated to the last 5000 lines.
  inflating: Corn/Train/Common Rust/RS_Rust 1593_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1594.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1594_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1595.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1595_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1596_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1597.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1597_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1598.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1599.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1600.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1600_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1601.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1601_flipLR.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1602.JPG  
  inflating: Corn/Train/Common Rust/RS_Rust 1602_flipLR.JPG  
  inflating: 

In [ ]:
import tensorflow as tf
import numpy as np
import os

from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, accuracy_score

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.19.0


In [ ]:
base_path = "/content/Corn"

train_dir = os.path.join(base_path, "Train")
val_dir   = os.path.join(base_path, "Val")
test_dir  = os.path.join(base_path, "Test")

In [ ]:
img_size = (260, 260)
batch_size = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

Found 5457 files belonging to 3 classes.
Found 1227 files belonging to 3 classes.
Found 141 files belonging to 3 classes.
Classes: ['Cercospora Leaf Spot', 'Common Rust', 'Northern Leaf Blight']


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def prepare(ds, training=False):
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare(train_ds, training=True)
val_ds   = prepare(val_ds, training=False)
test_ds  = prepare(test_ds, training=False)

In [ ]:
base_model = EfficientNetB2(
    include_top=False,
    input_shape=(260, 260, 3),
    weights="imagenet"
)

base_model.trainable = False  # Stage 1

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb2 (Functional)     │ (None, 9, 9, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         4,227 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,772,796 (29.65 MB)

 Trainable params: 4,227 (16.51 KB)

 Non-trainable params: 7,768,569 (29.63 MB)

In [ ]:
print("Stage 1: Training classifier head...")

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Stage 1: Training classifier head...
Epoch 1/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 169s 367ms/step - accuracy: 0.8064 - loss: 0.6288 - val_accuracy: 0.9299 - val_loss: 0.4421
Epoch 2/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 76s 222ms/step - accuracy: 0.9391 - loss: 0.4220 - val_accuracy: 0.9315 - val_loss: 0.4424
Epoch 3/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 74s 214ms/step - accuracy: 0.9478 - loss: 0.4134 - val_accuracy: 0.9381 - val_loss: 0.4191
Epoch 4/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 75s 217ms/step - accuracy: 0.9528 - loss: 0.4015 - val_accuracy: 0.9511 - val_loss: 0.3970
Epoch 5/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 74s 216ms/step - accuracy: 0.9527 - loss: 0.3997 - val_accuracy: 0.9332 - val_loss: 0.4311


In [ ]:
base_model.trainable = True

# Freeze lower layers, unfreeze top 50 layers
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.2),
    ModelCheckpoint("best_effnetb2_corn.h5", save_best_only=True)
]

print("Stage 2: Fine-tuning...")

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Stage 2: Fine-tuning...
Epoch 1/10
342/342 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step - accuracy: 0.9200 - loss: 0.4786

342/342 ━━━━━━━━━━━━━━━━━━━━ 159s 333ms/step - accuracy: 0.9200 - loss: 0.4785 - val_accuracy: 0.9601 - val_loss: 0.3963 - learning_rate: 1.0000e-05
Epoch 2/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9493 - loss: 0.4236

342/342 ━━━━━━━━━━━━━━━━━━━━ 77s 225ms/step - accuracy: 0.9493 - loss: 0.4235 - val_accuracy: 0.9625 - val_loss: 0.3870 - learning_rate: 1.0000e-05
Epoch 3/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.9547 - loss: 0.4149

342/342 ━━━━━━━━━━━━━━━━━━━━ 77s 224ms/step - accuracy: 0.9547 - loss: 0.4148 - val_accuracy: 0.9650 - val_loss: 0.3810 - learning_rate: 1.0000e-05
Epoch 4/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.9622 - loss: 0.4001

342/342 ━━━━━━━━━━━━━━━━━━━━ 75s 220ms/step - accuracy: 0.9622 - loss: 0.4001 - val_accuracy: 0.9698 - val_loss: 0.3724 - learning_rate: 1.0000e-05
Epoch 5/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.9595 - loss: 0.3994

...

342/342 ━━━━━━━━━━━━━━━━━━━━ 77s 225ms/step - accuracy: 0.9677 - loss: 0.3893 - val_accuracy: 0.9739 - val_loss: 0.3633 - learning_rate: 1.0000e-05
Epoch 7/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.9652 - loss: 0.3890

342/342 ━━━━━━━━━━━━━━━━━━━━ 77s 224ms/step - accuracy: 0.9652 - loss: 0.3890 - val_accuracy: 0.9747 - val_loss: 0.3605 - learning_rate: 1.0000e-05
Epoch 8/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.9624 - loss: 0.3855

342/342 ━━━━━━━━━━━━━━━━━━━━ 81s 221ms/step - accuracy: 0.9624 - loss: 0.3855 - val_accuracy: 0.9747 - val_loss: 0.3588 - learning_rate: 1.0000e-05
Epoch 9/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9692 - loss: 0.3799

342/342 ━━━━━━━━━━━━━━━━━━━━ 77s 223ms/step - accuracy: 0.9692 - loss: 0.3798 - val_accuracy: 0.9723 - val_loss: 0.3567 - learning_rate: 1.0000e-05
Epoch 10/10
341/342 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.9720 - loss: 0.3757

342/342 ━━━━━━━━━━━━━━━━━━━━ 81s 221ms/step - accuracy: 0.9720 - loss: 0.3756 - val_accuracy: 0.9764 - val_loss: 0.3541 - learning_rate: 1.0000e-05


In [ ]:
y_true, y_pred = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

print("\nFINAL TEST RESULTS:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


FINAL TEST RESULTS:
Accuracy: 0.9858156028368794
                      precision    recall  f1-score   support

Cercospora Leaf Spot       0.98      1.00      0.99        45
         Common Rust       0.98      1.00      0.99        48
Northern Leaf Blight       1.00      0.96      0.98        48

            accuracy                           0.99       141
           macro avg       0.99      0.99      0.99       141
        weighted avg       0.99      0.99      0.99       141



In [ ]:
model.save("effnetb2_corn_final.h5")
print("Model saved successfully.")
model_size = os.path.getsize("effnetb2_corn_final.h5") / (1024 * 1024)
print(f"Model Size: {model_size:.2f} MB")
from google.colab import files
files.download("effnetb2_corn_final.h5")

Model saved successfully.
Model Size: 62.25 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>